# Paso 4 - Clustering no supervisado sobre los candidatos

**Regla de esta línea de trabajo:** sin etiquetas, sin motor viejo (ver
`README.md`). Este notebook prueba si los 623 candidatos del Paso 3 tienen
**estructura natural** que separe alimentación/servido/ruido, usando solo
features ya construidas en esta carpeta (duración, `delta_neto_real`,
`max_abs_delta_g`, `n_lecturas`) más una nueva (`n_cambios_signo`, cuenta
cuántas veces se invierte la dirección del peso dentro del segmento —
detecta directamente el patrón de oscilación/manipulación ya visto a ojo en
la revisión manual).

**KPCL0034 y KPCL0035 se clusterizan por separado** — misma regla de todo
Investigacion_v2, gramos de comida y de agua no son la misma magnitud.

Carga desde `data/lecturas_limpias.csv` (cache de Paso 1) -- **correr
`01_caracterizacion_fondo.ipynb` primero** si el cache no existe todavía.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
CACHE_CSV = NOTEBOOK_DIR / "data" / "lecturas_limpias.csv"
GAP_CUTOFF_S = 300
K_MARGEN = 5

# --- Carga desde cache + reconstruccion de segmentos (identico a Paso 3, con la
# correccion del Hallazgo 6: tolera 1 lectura aislada de fondo) ------------------
df = pd.read_csv(CACHE_CSV)
df["device_id"] = df["device_id"].astype("category")
df["device_code"] = df["device_code"].astype("category")
df["ts"] = pd.to_datetime(df["ts"], format="ISO8601", utc=True)

df["delta_peso"] = df.groupby("device_id", observed=True)["peso"].diff()
df["delta_t"] = df.groupby("device_id", observed=True)["ts"].diff().dt.total_seconds()
df["abs_delta_peso"] = df["delta_peso"].abs()
is_gap = df["delta_t"] > GAP_CUTOFF_S
paso_estable = (df["delta_peso"] == 0) & (~is_gap.fillna(False))

cambio = (paso_estable != paso_estable.shift(1)) | (df["device_code"] != df["device_code"].shift(1))
racha_id = cambio.cumsum()
racha_len = df.groupby(racha_id)["peso"].transform("size")
es_corte_real = is_gap.fillna(False) | df["delta_peso"].isna() | (paso_estable & (racha_len >= 2))
es_movimiento = ~es_corte_real
cid = es_corte_real.groupby(df["device_code"], observed=True).cumsum()

DEVICE_CODES = df["device_code"].cat.categories
filas = []
for _code in DEVICE_CODES:
    _mask = es_movimiento & (df["device_code"] == _code)
    _grp = df.loc[_mask].groupby(cid[_mask], observed=True)
    _n_signos = _grp["delta_peso"].apply(lambda s: (np.sign(s).diff().fillna(0) != 0).sum())
    _seg = pd.DataFrame({
        "device_code": _code, "n_lecturas": _grp.size(), "duracion_s": _grp["delta_t"].sum(),
        "delta_neto_g": _grp["delta_peso"].sum(), "max_abs_delta_g": _grp["abs_delta_peso"].max(),
        "n_cambios_signo": _n_signos,
        "idx_inicio": _grp.apply(lambda g: g.index.min()), "idx_fin": _grp.apply(lambda g: g.index.max()),
    })
    filas.append(_seg)
segmentos_df = pd.concat(filas, ignore_index=True)

# nivel antes/despues (K_MARGEN lecturas paso_estable) -- igual que Paso 3
niveles_antes, niveles_despues = [], []
for _code, _sub in segmentos_df.groupby("device_code", observed=True):
    _idx_estable = np.array(sorted(df.index[(df["device_code"] == _code) & paso_estable].tolist()))
    for _, _row in _sub.iterrows():
        _pi = np.searchsorted(_idx_estable, _row["idx_inicio"])
        _antes = _idx_estable[max(0, _pi - K_MARGEN):_pi]
        _pf = np.searchsorted(_idx_estable, _row["idx_fin"], side="right")
        _despues = _idx_estable[_pf:_pf + K_MARGEN]
        niveles_antes.append(df.loc[_antes, "peso"].median() if len(_antes) == K_MARGEN else np.nan)
        niveles_despues.append(df.loc[_despues, "peso"].median() if len(_despues) == K_MARGEN else np.nan)
segmentos_df["nivel_antes"] = niveles_antes
segmentos_df["nivel_despues"] = niveles_despues
segmentos_df["delta_neto_real"] = segmentos_df["nivel_despues"] - segmentos_df["nivel_antes"]
segmentos_df["ts_inicio"] = df.loc[segmentos_df["idx_inicio"], "ts"].reset_index(drop=True)
segmentos_df["ts_fin"] = df.loc[segmentos_df["idx_fin"], "ts"].reset_index(drop=True)

# candidatos (mismo criterio z-score modificado del Paso 3)
segmentos_df["es_candidato"] = False
for _code, _g in segmentos_df.groupby("device_code", observed=True):
    _mediana = _g["max_abs_delta_g"].median()
    _mad = (_g["max_abs_delta_g"] - _mediana).abs().median()
    _umbral = _mediana + (3.5 / 0.6745) * _mad
    segmentos_df.loc[_g.index, "es_candidato"] = _g["max_abs_delta_g"] > _umbral

candidatos_df = segmentos_df[segmentos_df["es_candidato"]].dropna(subset=["delta_neto_real"]).copy()
# codigo unico por candidato -- device_code + timestamp de inicio (con precision de
# microsegundos, ya suficiente para que dos candidatos del mismo device nunca choquen)
candidatos_df["candidato_id"] = (
    candidatos_df["device_code"].astype(str) + "_" +
    candidatos_df["ts_inicio"].dt.strftime("%Y%m%d%H%M%S%f")
)
print(f"Candidatos con contexto completo: {len(candidatos_df):,}")
print(candidatos_df.groupby("device_code", observed=True).size())


## Features para clustering

`duracion_s` y `max_abs_delta_g` van con log (`log1p`) porque su distribución
es muy asimétrica (Paso 3) — sin eso, los pocos valores extremos dominan la
distancia euclidiana y opacan al resto. `delta_neto_real` y `n_cambios_signo`
van sin transformar (ya son razonablemente simétricos / acotados). Todo se
estandariza (media 0, desvío 1) antes de clusterizar — si no, `duracion_s`
(escala ~cientos) dominaría sobre `n_cambios_signo` (escala ~decenas) sin
que eso sea una decisión real, sería un artefacto de unidades.


In [ ]:
from sklearn.preprocessing import StandardScaler

FEATURES_BASE = ["duracion_s", "delta_neto_real", "max_abs_delta_g", "n_lecturas", "n_cambios_signo"]

def construir_matriz(df_candidatos):
    X = df_candidatos[FEATURES_BASE].copy()
    X["duracion_s"] = np.log1p(X["duracion_s"])
    X["max_abs_delta_g"] = np.log1p(X["max_abs_delta_g"])
    return StandardScaler().fit_transform(X)


## Correr varios modelos (por device_code)

`k=3` en KMeans/Agglomerative/GMM es una hipótesis a favor (calza con
alimentación/servido/ruido) — el silhouette score dice si esa hipótesis
tiene sustento en los datos o no, no se asume de entrada. DBSCAN no fija
`k` — encuentra sus propios clusters y marca outliers como ruido (`-1`),
útil como chequeo cruzado, aunque "ruido" en sentido DBSCAN es solo
"outlier estadístico", no necesariamente la categoría de dominio "ruido".


In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

# corre los 4 modelos POR device_code (nunca mezclados) y guarda las etiquetas
# directo en candidatos_df -- deja todo en un solo frame, listo para exportar
for _col in ["cluster_kmeans", "cluster_agg", "cluster_gmm", "cluster_dbscan"]:
    candidatos_df[_col] = -99

for _code in DEVICE_CODES:
    _sub = candidatos_df[candidatos_df["device_code"] == _code]
    _X = construir_matriz(_sub)

    _km = KMeans(n_clusters=3, random_state=0, n_init=10).fit(_X)
    _agg = AgglomerativeClustering(n_clusters=3).fit(_X)
    _gm = GaussianMixture(n_components=3, random_state=0).fit(_X)
    _gm_labels = _gm.predict(_X)
    _db = DBSCAN(eps=1.0, min_samples=5).fit(_X)

    candidatos_df.loc[_sub.index, "cluster_kmeans"] = _km.labels_
    candidatos_df.loc[_sub.index, "cluster_agg"] = _agg.labels_
    candidatos_df.loc[_sub.index, "cluster_gmm"] = _gm_labels
    candidatos_df.loc[_sub.index, "cluster_dbscan"] = _db.labels_

    print(f"--- {_code} ---")
    for _nombre, _labels in [("KMeans", _km.labels_), ("Agglomerative", _agg.labels_), ("GMM", _gm_labels)]:
        print(f"  {_nombre}: tamanos={np.bincount(_labels).tolist()}, "
              f"silhouette={silhouette_score(_X, _labels):.3f}")
    _n_clusters_db = len(set(_db.labels_)) - (1 if -1 in _db.labels_ else 0)
    print(f"  DBSCAN (eps=1.0): {_n_clusters_db} clusters, {(_db.labels_ == -1).sum()} outliers")

cand34 = candidatos_df[candidatos_df["device_code"] == "KPCL0034"].reset_index(drop=True)
print()
print("Acuerdo KMeans vs Agglomerative (KPCL0034):")
print(pd.crosstab(cand34["cluster_kmeans"], cand34["cluster_agg"]))


## Interpretar los clusters (KMeans, k=3)


In [ ]:
print(cand34.groupby("cluster_kmeans")[FEATURES_BASE].median().round(2))

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
for _c in sorted(cand34["cluster_kmeans"].unique()):
    _sub = cand34[cand34["cluster_kmeans"] == _c]
    ax.scatter(_sub["duracion_s"], _sub["delta_neto_real"], label=f"cluster {_c}", alpha=0.6)
ax.set_xscale("log")
ax.set_xlabel("duracion_s (log)")
ax.set_ylabel("delta_neto_real (g)")
ax.axhline(0, color="gray", linewidth=0.5)
ax.legend()
ax.set_title("KPCL0034 - clusters KMeans (k=3)")
plt.show()


## Cruzar contra las etiquetas manuales ya hechas

Los 5 candidatos ya etiquetados a mano como "ruido" (revisión manual,
notebook 04) — ¿en qué cluster caen?


In [ ]:
LABELS_CSV = NOTEBOOK_DIR / "data" / "candidatos_etiquetas_manuales.csv"
if LABELS_CSV.exists():
    etiquetas_df = pd.read_csv(LABELS_CSV)
    etiquetas_df["ts_inicio"] = pd.to_datetime(etiquetas_df["ts_inicio"], format="ISO8601", utc=True)
    _cruce = etiquetas_df.merge(
        cand34[["ts_inicio", "cluster_kmeans", "duracion_s", "delta_neto_real", "n_cambios_signo"]],
        on="ts_inicio", how="left",
    )
    print(_cruce)
else:
    print("Todavia no hay etiquetas manuales guardadas.")


## Ejemplos reales de cada cluster (revisión visual)


In [ ]:
MARGEN_GRAFICO_LECTURAS = 10

def graficar_fila(fila, device_code, ax=None):
    _m = df["device_code"] == device_code
    _ini = max(df.loc[_m].index.min(), fila["idx_inicio"] - MARGEN_GRAFICO_LECTURAS)
    _fin = min(df.loc[_m].index.max(), fila["idx_fin"] + MARGEN_GRAFICO_LECTURAS)
    _ventana = df.loc[_m].loc[_ini:_fin]
    _ax = ax or plt.subplots(figsize=(5, 2.5))[1]
    _ax.plot(_ventana["ts"], _ventana["peso"], marker="o", markersize=3)
    _ax.axvspan(fila["ts_inicio"], fila["ts_fin"], color="orange", alpha=0.2)
    _ax.set_title(
        f"cluster {fila['cluster_kmeans']} | {fila['ts_inicio']:%m-%d %H:%M} | "
        f"dur={fila['duracion_s']:.0f}s neto={fila['delta_neto_real']:.0f}g",
        fontsize=8,
    )
    _ax.tick_params(axis="x", labelrotation=20, labelsize=7)
    return _ax

N_EJEMPLOS = 4
fig, axes = plt.subplots(3, N_EJEMPLOS, figsize=(4 * N_EJEMPLOS, 7.5))
for _fila_idx, _c in enumerate(sorted(cand34["cluster_kmeans"].unique())):
    _muestra = cand34[cand34["cluster_kmeans"] == _c].sample(
        min(N_EJEMPLOS, (cand34['cluster_kmeans'] == _c).sum()), random_state=0,
    )
    for _col_idx, (_, _fila) in enumerate(_muestra.iterrows()):
        graficar_fila(_fila, "KPCL0034", ax=axes[_fila_idx, _col_idx])
fig.tight_layout()
plt.show()


## Exportar resultado del clustering (para la app de visualización)

`data/candidatos_clusters.csv` — un candidato por fila, con su `candidato_id`
único, contexto (antes/después), y la etiqueta de los 4 modelos. Lo consume
`visualizacion/app_candidatos.py`.


In [ ]:
COLUMNAS_EXPORT = [
    "candidato_id", "device_code", "idx_inicio", "idx_fin", "ts_inicio", "ts_fin",
    "duracion_s", "n_lecturas", "delta_neto_g", "delta_neto_real", "max_abs_delta_g",
    "n_cambios_signo", "nivel_antes", "nivel_despues",
    "cluster_kmeans", "cluster_agg", "cluster_gmm", "cluster_dbscan",
]
CLUSTERS_CSV = NOTEBOOK_DIR / "data" / "candidatos_clusters.csv"
candidatos_df[COLUMNAS_EXPORT].sort_values(["device_code", "ts_inicio"]).to_csv(CLUSTERS_CSV, index=False)
print(f"Exportado: {CLUSTERS_CSV} ({len(candidatos_df):,} candidatos)")


## Cierre

**Resultado real (2026-08-29):** KMeans/Agglomerative con `k=3` dan silhouette
~0.44/0.42 en KPCL0034 (490 candidatos) — estructura real, no trivial. Los 3
clusters (medianas): uno corto+limpio+sin reversión (candidato a servido),
uno largo+oscilante (candidato a alimentación, no puro), uno corto+chico con
vaivén leve (ambiguo).

**Hallazgo — mismo punto ciego que el motor viejo:** al cruzar contra los 5
candidatos ya etiquetados a mano como "ruido" (notebook 04), **3 de 5** (el
cluster oscilante 03:15-03:19, que vos mismo identificaste como manipulación)
caen en el cluster "servido limpio" — porque `n_cambios_signo` solo cuenta
reversiones **dentro** de un segmento, y esos 3 son segmentos *separados* que
oscilan *entre sí*, no dentro de cada uno. Inspección visual de ejemplos
aleatorios del mismo cluster confirma que mezcla saltos limpios reales con
caídas-y-recuperación rápidas (V corta, neto≈0) — no es un cluster puro.

**Conclusión:** dos líneas de evidencia independientes (esto y la prueba con
el motor viejo, ver historial) apuntan a la misma causa — hace falta juntar
candidatos pegados en el tiempo **antes** de analizar la forma, no solo
mejorar las features de cada segmento aislado. Pendiente, no resuelto acá.
